# Lab - PII and Secrets: Send Only What Is Needed

Three tasks on what leaves your server. Every request to Claude leaves your building, and what is in it is your responsibility - the customer's personal data, and your own keys.

| Task | What you learn |
| --- | --- |
| 1 | Redact email, card and phone before the request; keep the order number |
| 2 | A boundary check that refuses to send anything a pattern still matches |
| 3 | Secrets from the environment, masked in every dump, ignored by git |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import fnmatch
import json
import os
import re
from pathlib import Path

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

message = (
    "Hi, I am alex@example.com, phone +353 85 123 4567. "
    "I paid with card 4111 1111 1111 1111 for order "
    "ORD-12345678 and want to return the jacket."
)

# One pattern per kind of personal data. The order matters: a card number also
# looks like a phone number, so the card pattern runs first.
PATTERNS = {
    "EMAIL": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "CARD": r"\b(?:\d[ -]?){12,15}\d\b",
    "PHONE": r"\+?\d[\d -]{8,}\d",
}

SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Personal data arrives as placeholders such as <EMAIL>; "
    "never ask the customer to send it. Answer in two sentences."
)

# What the platform's secret store puts in the environment, and the project's
# .gitignore - both as a real deployment would have them.
os.environ.setdefault("SHOPASSIST_WEBHOOK_SECRET", "whsec_lab_7f3a9c2e41")
Path(".gitignore").write_text(".env\n__pycache__/\n*.log\n")

---

## Task 1 - Redact before the request

In [ ]:
# ============================================================
# TASK 1 - Redact before the request
# ============================================================
#
# WHAT TO DO
#   Finish redact(): replace every match of every pattern with
#   a placeholder in angle brackets - <EMAIL>, <CARD>, <PHONE>.
#   Print what would be sent.
#
# WHY IT MATTERS
#   Every request to Claude leaves your building, and what is in
#   it is your responsibility. ShopAssist needs the order number.
#   It does not need the email, the phone or the card number to
#   answer - and those are personal data. The rule is simple:
#   what the model does not need, the model does not get.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Three patterns, one per kind of data".
#
# HOW TO DO IT
#   Three blanks:
#
#       PATTERNS.items()   every name and its pattern, in order
#       f"<{name}>"        the placeholder for that kind of data
#       text               the redacted text
# ============================================================

try:
    # BEGIN SOLUTION
    def redact(text):
        for name, pattern in PATTERNS.items():
            text = re.sub(pattern, f"<{name}>", text)
        return text
    # SCAFFOLD: def redact(text):
    # SCAFFOLD:     for name, pattern in ...:   # PATTERNS.items()
    # SCAFFOLD:         text = re.sub(pattern, ..., text)   # f"<{name}>"
    # SCAFFOLD:     return ...   # text
    # END SOLUTION: replace each ... below with the value named beside it

    safe = redact(message)
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc) or "must be str" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print("original:", message)
print("redacted:", safe)

check("redact_text", redact=redact)

---

## Task 2 - Refuse to send what still leaks

In [ ]:
# ============================================================
# TASK 2 - Refuse to send what still leaks
# ============================================================
#
# WHAT TO DO
#   Write send(): before a text goes to Claude, check it once
#   more with every pattern. If anything still matches, raise
#   PIILeak and send nothing. Otherwise record it and send it.
#   Send the redacted message.
#
# WHY IT MATTERS
#   Redaction runs on text it has never seen; one day a pattern
#   misses, or someone calls the API without redacting first. A
#   last check at the boundary that fails closed - refuse, do not
#   send - turns that day into an error in your logs instead of a
#   leak. Claude still answers: it never needed the data, and the
#   system prompt tells it not to ask for it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "We print what is actually sent, then we send it".
#
# HOW TO DO IT
#   Three blanks:
#
#       PATTERNS.values()   every pattern, once more
#       text                the text that would leave
#       safe                the redacted message from Task 1
# ============================================================


class PIILeak(Exception):   # provided
    """Personal data was about to leave in a request."""


SENT = []   # provided: everything that actually left

try:
    # BEGIN SOLUTION
    def send(text):
        for pattern in PATTERNS.values():
            if re.search(pattern, text):
                raise PIILeak("personal data still in the request - nothing sent")
        SENT.append(text)
        response = client.messages.create(
            model=MODEL, max_tokens=200, output_config={"effort": "low"},
            system=SYSTEM, messages=[{"role": "user", "content": text}])
        return response.content[0].text

    print("sent to Claude:", safe)
    print(send(safe))
    # SCAFFOLD: def send(text):
    # SCAFFOLD:     for pattern in ...:   # PATTERNS.values()
    # SCAFFOLD:         if re.search(pattern, ...):   # text
    # SCAFFOLD:             raise PIILeak("personal data still in the request - nothing sent")
    # SCAFFOLD:     SENT.append(text)
    # SCAFFOLD:     response = client.messages.create(
    # SCAFFOLD:         model=MODEL, max_tokens=200, output_config={"effort": "low"},
    # SCAFFOLD:         system=SYSTEM, messages=[{"role": "user", "content": text}])
    # SCAFFOLD:     return response.content[0].text
    # SCAFFOLD:
    # SCAFFOLD: print("sent to Claude:", safe)
    # SCAFFOLD: print(send(...))   # safe
    # END SOLUTION: replace each ... below with the value named beside it
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc) or "must be str" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("fail_closed", send=send, sent=SENT, leak_error=PIILeak)

---

## Task 3 - Keys: in the environment, ignored, never printed

In [ ]:
# ============================================================
# TASK 3 - Keys: in the environment, ignored, never printed
# ============================================================
#
# WHAT TO DO
#   Build the app's config from the environment, then write two
#   helpers: safe_view(), which masks every secret before the
#   config is printed or logged (a name ending in key, secret,
#   token or password - max_tokens is not a secret), and
#   ignored(), which checks a
#   path against the project's .gitignore.
#
# WHY IT MATTERS
#   The other secret in the folder is the key. Three habits: it
#   lives in the environment, never in the code; the .env file
#   is ignored by git, so it cannot be committed by mistake; and
#   nothing prints it - no log line, no error, no debug dump. In
#   production the same code reads the platform's secret store.
#   Once a key is in the history, the only fix is to revoke it.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now the other secret in this folder: the API key".
#
# HOW TO DO IT
#   Three blanks:
#
#       r"(?:^|_)(?:key|secret|token|password)$"   names ending in a secret word
#       value[-4:]                     the only part a log may show
#       fnmatch.fnmatch                does the path match an ignore line
# ============================================================

CONFIG = {   # provided: values come from the environment, never from the code
    "model": MODEL,
    "max_tokens": 200,
    "webhook_secret": os.environ.get("SHOPASSIST_WEBHOOK_SECRET"),
}

try:
    # BEGIN SOLUTION
    def safe_view(config):
        view = {}
        for name, value in config.items():
            if re.search(r"(?:^|_)(?:key|secret|token|password)$", name, re.I) and value:
                view[name] = "****" + str(value)[-4:]
            else:
                view[name] = value
        return view

    def ignored(path):
        rules = [line.strip() for line in Path(".gitignore").read_text().splitlines()
                 if line.strip() and not line.startswith("#")]
        return any(fnmatch.fnmatch(path, rule) for rule in rules)
    # SCAFFOLD: def safe_view(config):
    # SCAFFOLD:     view = {}
    # SCAFFOLD:     for name, value in config.items():
    # SCAFFOLD:         if re.search(..., name, re.I) and value:   # r"(?:^|_)(?:key|secret|token|password)$"
    # SCAFFOLD:             view[name] = "****" + str(value)[...]   # -4: - the last four only
    # SCAFFOLD:         else:
    # SCAFFOLD:             view[name] = value
    # SCAFFOLD:     return view
    # SCAFFOLD:
    # SCAFFOLD: def ignored(path):
    # SCAFFOLD:     rules = [line.strip() for line in Path(".gitignore").read_text().splitlines()
    # SCAFFOLD:              if line.strip() and not line.startswith("#")]
    # SCAFFOLD:     return any(fnmatch.fnmatch(path, ...) for rule in rules)   # rule
    # END SOLUTION: replace each ... below with the value named beside it

    print("config, as a log may show it:", safe_view(CONFIG))
    print(".env ignored by git:", ignored(".env"), "| redact.py ignored:", ignored("redact.py"))
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc) or "must be str" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("secrets_handling", safe_view=safe_view, ignored=ignored, config=CONFIG)

---

## Done

Redact personal data before the request and send only what the task needs - with a check at the boundary that fails closed. Keep keys in the environment, ignored by git, never printed; revoke one the moment it leaks.